# Bombyx mori piRNA smRNA-seq analysis

This notebook mainly records the smRNA-seq analysis methods for samples after knockout.

## Notebook Configuration

In [ ]:
from pathlib import Path

from bm_pirna.config import RAW_DATA_DIR,INTERIM_DATA_DIR,REPORTS_DIR,EXTERNAL_DATA_DIR,PROCESSED_DATA_DIR
from bm_pirna.utils import run_cmd

In [ ]:
task_name = "smRNA-seq_20260130"

#qc
sample_dir = RAW_DATA_DIR / "RNPS1-DHX15-SUGP1/dm_smRNA-seq"
smrna_adapter_file = EXTERNAL_DATA_DIR / "sample_adapter.csv"
qc_output = INTERIM_DATA_DIR / task_name / "qc"
qc_report_dir = REPORTS_DIR / task_name / "qc_reports"

# piRNA analysis derived from transposons
# structure rna filter
structure_rna = EXTERNAL_DATA_DIR / "dm/dmel-all-miRNA-tRNA-snoRNA.fasta"
structure_rna_index = str(INTERIM_DATA_DIR / "bowtie_index/dm_structure_rna_index")
structure_rna_filtered_output = INTERIM_DATA_DIR / task_name / "dm_structure_rna_filtered"

# Conventional piRNA determined
transposon_fasta = EXTERNAL_DATA_DIR / "dm/dmel-all-transposon-r6.66.fasta"
transposon_index = str(INTERIM_DATA_DIR / "bowtie_index/dm_transposon_index")
transposon_filtered_output = INTERIM_DATA_DIR / task_name / "dm_transposon_filtered"

# stats
transposon_dedrived_pirna_stats_output = PROCESSED_DATA_DIR / task_name / "transposon_dedrived_pirna_stats"

# transposon rpm analysis
sample_map_file = EXTERNAL_DATA_DIR / "sample_map.csv"
transposon_rpm_output = PROCESSED_DATA_DIR / task_name / "transposon_rpm"

# ================================================================================

# transposon analysis derived from tRNA
# trna derived piRNA filter
trna_fasta = EXTERNAL_DATA_DIR / "dm/dmel-all-tRNA-r6.66.fasta"
trna_index = str(INTERIM_DATA_DIR / "bowtie_index/dm_trna_index")
trna_filtered_output = INTERIM_DATA_DIR / task_name / "dm_trna_filtered"

# td-piRNA variation analysis
sample_map_file = EXTERNAL_DATA_DIR / "sample_map.csv"
td_pirna_variation_output = PROCESSED_DATA_DIR / task_name / "dm_td_pirna_variation"

# stats
trna_drived_pirna_stats_output = PROCESSED_DATA_DIR / task_name / "dm_trna_drived_pirna_stats"
trna_drived_pirna_match_output = PROCESSED_DATA_DIR / task_name / "dm_trna_drived_pirna_match"

## Quality control
Remove low-quality reads and adapter sequences from raw sequencing data.

In [ ]:
# qc cmd
qc_cmd = f"pixi run python -m bm_pirna.smrna_seq.qc \
            -o {qc_output} -a {smrna_adapter_file} \
            -l 24-35 --cores 70 --pattern *.f*q.gz \
            --report-dir {qc_report_dir} {sample_dir}"

# run
run_cmd(
    qc_cmd,
    [qc_output, qc_report_dir],
    force=True
)

## piRNA analysis derived from transposons

### Structural RNA filtering
Filtering out structural RNAs, such as tRNA, rRNA, snRNA, and snoRNA. The sequences of the structural RNAs of Bombyx mori is extracted from the [NCBI genome annotation](https://www.ncbi.nlm.nih.gov/refseq/annotation_euk/Bombyx_mori/GCF_030269925.1-RS_2024_01/).

In [ ]:
# cmd
structure_rna_index_cmd = f"pixi run bowtie-build -f --quiet {structure_rna} {structure_rna_index}"
structure_rna_fiter_cmd = f"pixi run python -m bm_pirna.smrna_seq.filter \
                            {qc_output}/24-35_length_filtered {structure_rna_index} \
                            -v 1 -m exclude --threads 16 \
                            --output-dir {structure_rna_filtered_output} \
                            --suffix .fq.gz"

# run
run_cmd(structure_rna_index_cmd,[Path(f"{structure_rna_index}.1.ebwt")])
run_cmd(structure_rna_fiter_cmd,[structure_rna_filtered_output])

### Transposon derived piRNA filtering

Retain only sequences that can be aligned to the transposon, with no mismatches allowed.

In [ ]:
# cmd
transposon_index_cmd = f"pixi run bowtie-build -f --quiet {transposon_fasta} {transposon_index}"
transposon_filter_cmd = f"pixi run python -m bm_pirna.smrna_seq.filter \
                        {structure_rna_filtered_output} \
                        {transposon_index} \
                        -v 0 -m include \
                        --threads 16 \
                        --output-dir {transposon_filtered_output} \
                        --suffix .transposon.fq.gz"

# run
run_cmd(transposon_index_cmd,[Path(f"{transposon_index}.1.ebwt")])
run_cmd(transposon_filter_cmd,[transposon_filtered_output])

### Reads stats
Statistics of the filtered sequence information

In [ ]:
# cmd
transposon_stats_cmd = f"python -m bm_pirna.smrna_seq.stats \
                        {transposon_filtered_output} \
                        -o {transposon_dedrived_pirna_stats_output}"

# run
run_cmd(transposon_stats_cmd, [transposon_dedrived_pirna_stats_output])

### Transposon RPM Analysis

In [ ]:
# cmd
transposon_rpm_cmd = f"python -m bm_pirna.smrna_seq.transposon_rpm \
    {transposon_dedrived_pirna_stats_output}/collapsed/ \
    -i {transposon_index} \
    -c {qc_report_dir}/cutadapt/cutadapt_summary.tsv \
    -s {sample_map_file} \
    -o {transposon_rpm_output} \
    -t 70 -m 1"

# run
run_cmd(transposon_rpm_cmd,[transposon_rpm_output])

## Transposon analysis derived from tRNA

### tRNA derived piRNA filtering
After obtaining piRNAs from the piRNA cluster, filter out the piRNAs derived from tRNA.

In [ ]:
# cmd
trna_index_cmd = f"pixi run bowtie-build -f --quiet {trna_fasta} {trna_index}"
trna_filter_cmd = f"pixi run python -m bm_pirna.smrna_seq.filter \
                    {qc_output}/24-35_length_filtered \
                    {trna_index} \
                    -v 0 -m include \
                    --threads 16 \
                    --output-dir {trna_filtered_output} \
                    --suffix .trna.fq.gz"

# run
run_cmd(trna_index_cmd,[Path(f"{trna_index}.1.ebwt")])
run_cmd(trna_filter_cmd,[trna_filtered_output])

### Reads stats
Statistics of the filtered sequence information

In [ ]:
# cmd
trna_stats_cmd = f"python -m bm_pirna.smrna_seq.stats {trna_filtered_output} -o {trna_drived_pirna_stats_output}"

# run
run_cmd(trna_stats_cmd, [trna_drived_pirna_stats_output])

### Find the piRNAs that can be matched to piRBase

In [ ]:
# cmd
trna_drived_pirna_match_cmd = f"pixi run python -m bm_pirna.smrna_seq.pirna_match \
                                {trna_drived_pirna_stats_output}/collapsed \
                                -o {trna_drived_pirna_match_output} \
                                -m 1 -f -t 75"

# run
run_cmd(trna_drived_pirna_match_cmd, [trna_drived_pirna_match_output],force=True)

### Analysis of piRNA Variations Derived from tRNA

In [ ]:
# cmd
td_pirna_variation_cmd = f"python -m bm_pirna.smrna_seq.pirna_rpm \
                            {trna_drived_pirna_match_output}/collapsed \
                            -c {qc_report_dir}/cutadapt/cutadapt_summary.tsv \
                            -s {sample_map_file} \
                            -o {td_pirna_variation_output}"

# run
run_cmd(td_pirna_variation_cmd,[td_pirna_variation_output])